# REPA × CIFAR-10 × MedDINOv3

Trains **SiT-S/8** on CIFAR-10 using **MedDINOv3 ViT-Base** as the representation encoder.

**Requirements (Kaggle datasets attached to this notebook):**
- `nikiboura/meddinov3` — contains MedDINOv3 code + checkpoint

All code runs from the cloned `nikiboura/REPA` fork. No local file edits.

## 1. Install dependencies

In [ ]:
%%bash
pip install -q accelerate diffusers timm einops

## 2. Clone REPA fork

In [ ]:
%%bash
cd /kaggle/working
if [ ! -d "REPA" ]; then
    git clone https://github.com/nikiboura/REPA.git REPA
fi
echo "REPA ready. Commit: $(git -C REPA rev-parse --short HEAD)"

## 3. Set paths

In [ ]:
import os

REPA_DIR   = '/kaggle/working/REPA'
DATA_DIR   = '/kaggle/working/data/cifar10_256'
MEDDINOV3  = '/kaggle/input/datasets/nikiboura/meddinov3-minimal/meddinov3_minimal'
CKPT_PATH  = '/kaggle/input/datasets/nikiboura/meddinov3-minimal/meddinov3_minimal/checkpoints/model.pth'

os.environ['MEDDINOV3_PATH'] = MEDDINOV3
os.environ['MEDDINOV3_CKPT'] = CKPT_PATH

os.makedirs(DATA_DIR, exist_ok=True)

print(f'MedDINOv3 exists : {os.path.isdir(MEDDINOV3)}')
print(f'Checkpoint exists: {os.path.isfile(CKPT_PATH)}')

## 4. Prepare CIFAR-10
Downloads CIFAR-10, resizes to 256×256, VAE-encodes each image, saves latents as `.npy`.

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, '/kaggle/working/REPA/prepare_cifar10.py',
    '--out-dir', '/kaggle/working/data/cifar10_256',
    '--resolution', '256',
    '--max-samples', '5000',
]
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
print('Exit code:', process.returncode)

## 5. Train
Runs 1000 steps as a quick correctness check. Checkpoints saved to `/kaggle/working/results/`.

In [ ]:
%%bash
cd /kaggle/working/REPA

MEDDINOV3_PATH=/kaggle/input/datasets/nikiboura/meddinov3-minimal/meddinov3_minimal \
MEDDINOV3_CKPT=/kaggle/input/datasets/nikiboura/meddinov3-minimal/meddinov3_minimal/checkpoints/model.pth \
accelerate launch \
    --mixed_precision fp16 \
    --num_processes 1 \
    train.py \
    --exp-name           repa-sits8-meddinov3-cifar10 \
    --output-dir         /kaggle/working/results \
    --report-to          none \
    --model              SiT-S/8 \
    --num-classes        10 \
    --encoder-depth      8 \
    --enc-type           meddinov3-vit-b \
    --data-dir           /kaggle/working/data/cifar10_256 \
    --resolution         256 \
    --batch-size         32 \
    --num-workers        2 \
    --epochs             1 \
    --max-train-steps    1000 \
    --checkpointing-steps 500 \
    --sampling-steps     10000 \
    --learning-rate      1e-4 \
    --mixed-precision    fp16 \
    --proj-coeff         0.5 \
    --cfg-prob           0.1 \
    --path-type          linear

## 6. Generate samples

In [ ]:
%%bash
cd /kaggle/working/REPA

CKPT=$(ls /kaggle/working/results/repa-sits8-meddinov3-cifar10/checkpoints/last.pt 2>/dev/null)
if [ -z "$CKPT" ]; then
    echo "ERROR: checkpoint not found. Did training finish?"
    exit 1
fi
echo "Using checkpoint: $CKPT"

torchrun --nproc_per_node=1 generate.py \
    --model              SiT-S/8 \
    --ckpt               $CKPT \
    --encoder-depth      8 \
    --num-classes        10 \
    --projector-embed-dims 768 \
    --per-proc-batch-size 32 \
    --num-fid-samples    256 \
    --path-type          linear \
    --mode               ode \
    --num-steps          50 \
    --cfg-scale          1.5 \
    --resolution         256

## 7. Show generated images

In [ ]:
import glob, os
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

sample_dirs = sorted(glob.glob('/kaggle/working/REPA/samples/SiT-S-8-last-*'))
if not sample_dirs:
    print('No samples found.')
else:
    sample_dir = sample_dirs[-1]
    pngs = sorted(glob.glob(os.path.join(sample_dir, '*.png')))[:16]
    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    for ax, p in zip(axes.flat, pngs):
        ax.imshow(np.array(Image.open(p)))
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'Showing samples from: {sample_dir}')